# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset ("Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells") using the `mlcroissant` library and referencing all resources by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL (as provided)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Show dataset metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")
print(f"Cite as: {dataset.metadata.cite_as}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values. All navigation will use `@id` for reference.

In [ ]:
# List all record sets and their fields using their @id
record_sets = dataset.metadata.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id}, name: {rs.name}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      Field @id: {field.id}, name: {field.name}, data_type: {field.data_type}")
    print("")

## 3. Data Extraction
Extract records from a specific record set. We'll use the main protein record set (by its `@id`).

**Note:** Replace the variables below with the actual `@id` values output above for your exploration.

In [ ]:
# List of chosen record_set @ids (copy the record set IDs from the overview section; update as needed)
# For demonstration, we'll collect all record set IDs found above
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Number of records: {len(df)}\n")
# For this demo, we'll proceed with the first record set
main_rs_id = record_set_ids[0] if record_set_ids else None
print(f"Using main record set @id for subsequent analysis: {main_rs_id}")
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
We will select a numeric field (by `@id`), perform filtering, normalization, and some grouping operations. Please replace field `@id`s with those you found in section 2 for your specific analysis.

*Examples below pick the most likely numeric and categorical fields. Adjust as needed for your exploration based on field availability and names.*

In [ ]:
df = dataframes[main_rs_id]

# Pick a numeric field by its column name (from previous output, e.g., 'coverage', or whatever is available/appropriate)
# Replace with correct field name or @id/column
available_numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
if available_numeric_cols:
    numeric_field = available_numeric_cols[0]  # Pick first numeric column
else:
    raise ValueError("No numeric columns found for EDA.")

print(f"Using numeric field: {numeric_field}")

# Filter records with numeric_field > threshold
threshold = df[numeric_field].mean() if df[numeric_field].notnull().sum() > 0 else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical/text field (pick first non-numeric field)
available_cat_cols = [c for c in df.columns if df[c].dtype == object and c != numeric_field]
if available_cat_cols:
    group_field = available_cat_cols[0]
    print(f"\nGrouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(grouped_df.head())
else:
    print("\nNo non-numeric fields to group by.")

## 5. Visualization
Visualize the distribution of the selected numeric field and the grouped means (when available).

In [ ]:
import matplotlib.pyplot as plt

# Plot histogram for the numeric field
plt.figure(figsize=(7,4))
df[numeric_field].hist(bins=30)
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.title(f'Distribution of {numeric_field}')
plt.show()

# If grouped_df exists, plot group means
if 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(10, 4))
    grouped_df.plot(x=group_field, y=numeric_field, kind='bar')
    plt.ylabel(f'Mean {numeric_field}')
    plt.title(f'Mean {numeric_field} by {group_field}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to:

- Load FAIR² data using the `mlcroissant` library referencing all resources by `@id`
- Survey available record sets and fields
- Extract tabular data for analysis
- Perform basic exploratory data analysis (filtering, normalization, and grouping)
- Visualize distributions and group summaries

Use the specific `@id` values in your exploration for deeper, customized analysis. Refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/api/) for advanced use.